In [1]:
#importing Libraries
import pandas as pd
import numpy as np


In [2]:
#Importing preprocessor objects
import joblib
X_train_cls_preprocessed = joblib.load("D:\\Project\\Desktop Files\\Clickstream\\Notebooks\\X_train_cls_preprocessed.pkl")
X_test_cls_preprocessed = joblib.load("D:\\Project\\Desktop Files\\Clickstream\\Notebooks\\X_test_cls_preprocessed.pkl")
X_train_reg_preprocessed = joblib.load("D:\\Project\\Desktop Files\\Clickstream\\Notebooks\\X_train_reg_preprocessed.pkl")
X_test_reg_preprocessed = joblib.load("D:\\Project\\Desktop Files\\Clickstream\\Notebooks\\X_test_reg_preprocessed.pkl")
y_train_cls = joblib.load("D:\\Project\\Desktop Files\\Clickstream\\Notebooks\\y_train_cls.pkl")
y_test_cls = joblib.load("D:\\Project\\Desktop Files\\Clickstream\\Notebooks\\y_test_cls.pkl")
y_train_reg = joblib.load("D:\\Project\\Desktop Files\\Clickstream\\Notebooks\\y_train_reg.pkl")
y_test_reg = joblib.load("D:\\Project\\Desktop Files\\Clickstream\\Notebooks\\y_test_reg.pkl")

preprocessor = joblib.load("D:\\Project\\Desktop Files\\Clickstream\\Notebooks\\preprocessor.pkl")



In [3]:
#Importing models

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.cluster import KMeans, DBSCAN
from xgboost import XGBClassifier, XGBRegressor

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)

import numpy as np
import mlflow
import mlflow.sklearn

In [4]:
#SET MLflow EXPERIMENT

mlflow.set_experiment("Clickstream_Customer_Conversion_Project")

<Experiment: artifact_location='file:d:/Project/Desktop Files/Clickstream/Notebooks/mlruns/1', creation_time=1780639607854, experiment_id='1', last_update_time=1780639607854, lifecycle_stage='active', name='Clickstream_Customer_Conversion_Project', tags={}, workspace='default'>

In [ ]:
#Classification Evaluation Function

def evaluate_classification(model, X_train, X_test, y_train, y_test, model_name):

    with mlflow.start_run(run_name=model_name):

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        # ROC-AUC safe handling
        try:
            y_prob = model.predict_proba(X_test)[:,1]
            roc_auc = roc_auc_score(y_test, y_prob)
        except:
            roc_auc = None

        mlflow.log_param("model", model_name)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1", f1)

        if roc_auc:
            mlflow.log_metric("roc_auc", roc_auc)

        mlflow.sklearn.log_model(model, model_name)

        print(model_name)
        print("Accuracy:", accuracy)
        print("F1:", f1)


In [ ]:
# Regression Evaluation Function

def evaluate_regression(model, X_train, X_test, y_train, y_test, model_name):

    with mlflow.start_run(run_name=model_name):

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)

        mlflow.log_param("model", model_name)

        mlflow.log_metric("mae", mae)
        mlflow.log_metric("mse", mse)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2_score", r2)

        mlflow.sklearn.log_model(model, model_name)

        print(model_name)
        print("MAE:", mae)
        print("RMSE:", rmse)
        print("R2 Score:", r2)


In [10]:
models_reg = {
    "LinearRegression": LinearRegression(),

    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),

    "XGBoostRegressor": XGBRegressor(
        n_estimators=100,
        eval_metric="rmse",
        random_state=42
    ),

    "KNNRegressor": KNeighborsRegressor(n_neighbors=5)
}

In [11]:
for name, model in models_reg.items():

    evaluate_regression(
        model,
        X_train_reg_preprocessed,
        X_test_reg_preprocessed,
        y_train_reg,
        y_test_reg,
        name
    )

2026/06/07 00:18:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/07 00:18:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LinearRegression
MAE: 8.728816045065212
RMSE: 10.26284366000306
R2 Score: 0.33551296216632953


2026/06/07 00:21:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/07 00:21:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RandomForestRegressor
MAE: 0.0003100921589363953
RMSE: 0.04930262267372734
R2 Score: 0.9999846647404044


2026/06/07 00:22:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/07 00:22:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


XGBoostRegressor
MAE: 0.010251602157950401
RMSE: 0.017626985057210134
R2 Score: 0.9999980330467224


2026/06/07 00:25:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/07 00:25:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


KNNRegressor
MAE: 0.5191267563075993
RMSE: 2.4349453910220253
R2 Score: 0.9625950103679853


In [75]:
#Clustering

cluster_features = [
    "order",
    "page",
    "country_grouped",
    "page1_main_category",
    "colour_grouped",
    "location",
    "model_photography",
    "page2_clothing_model_freq",
    "day_grouped",
    "month"
]

In [79]:
train_df_fe=pd.read_csv("D:\\Project\\Desktop Files\\Clickstream\\Notebooks\\feature_engineered_data.csv")
train_df_fe.columns

Index(['month', 'day', 'order', 'page1_main_category', 'page2_clothing_model',
       'location', 'model_photography', 'price', 'price_2', 'page',
       'country_grouped', 'colour_grouped', 'day_grouped'],
      dtype='object')

In [81]:
#Applying freqiency encoder for page2_clothing_model

freq_map = train_df_fe["page2_clothing_model"].value_counts()

# Apply frequency encoding
train_df_fe["page2_clothing_model_freq"] = (
    train_df_fe["page2_clothing_model"]
    .map(freq_map)
)


In [82]:
X_cluster = train_df_fe[cluster_features]
X_cluster.columns

Index(['order', 'page', 'country_grouped', 'page1_main_category',
       'colour_grouped', 'location', 'model_photography',
       'page2_clothing_model_freq', 'day_grouped', 'month'],
      dtype='object')

In [84]:
X_cluster_pp = preprocessor.fit_transform(X_cluster)

In [87]:
X_cluster_pp

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1323790 stored elements and shape (132379, 47)>

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=42)
kmeans_labels = kmeans.fit_predict(X_cluster_pp)

from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=1.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_cluster_pp)

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

In [ ]:
print("KMeans Silhouette:",
      silhouette_score(X_cluster_scaled, kmeans_labels))

print("KMeans DB Index:",
      davies_bouldin_score(X_cluster_scaled, kmeans_labels))